In [ ]:
# 라이브러리 Import
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("="*60)

In [ ]:
# 1) 데이터 로드
df1 = pd.read_csv('product_type_1.csv',header=[0,1])

In [ ]:
df1.info()

In [ ]:
# 'defects'라는 상위 레벨만 따로 떼어낸 임시 데이터프레임 생성
defects_df = df1['defects'] 

# 2) 타겟/피처 분리
 # y: Defects 그룹 전체 (멀티라벨)
defect_groups = {
    "표면": [
        "dent_1",
        "stain_1",
        "exfoliation_1",
        "exfoliation_2"
    ],
    "구조": [
        "short_shot_1",
        "short_shot_2",
        "bubble_1",
        "bubble_2",
        "deformation_1",
        "deformation_2"
    ]
}


y = pd.DataFrame(index=df1.index)

for group_name, cols in defect_groups.items():
    y[group_name] = defects_df[cols].any(axis=1).astype(int)

X = df1[['process','sensor']].copy()
X = X.drop(columns=[('process','id'), ('process','product_type'), ('process','shot')])


# 5) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

display(y)
display(X)

In [ ]:
X_train.isna().sum()

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier

# 6) 모델 학습
model = MultiOutputClassifier(XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
))
model.fit(X_train, y_train)

# 7) 평가
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

# 평가 시에는 각 컬럼별로 리포트를 출력해야 합니다.
print("--- 표면 결함 리포트 ---")
print(classification_report(y_test['표면'], y_pred[:, 0]))

print("--- 구조 결함 리포트 ---")
print(classification_report(y_test['구조'], y_pred[:, 1]))